# FinLearn — AI-Powered Financial Learning Platform
## Detailed minor-project notebook: financial scoring + Random Forest + AdaBoost + XGBoost

**Reference materials used:** your original `minipro1.py` Colab workflow, your `Screener_Query_Results_Page1 (1).xlsx` workbook, and your supplied 10-week FinLearn project plan.

**Notebook purpose:** preserve the sequence and intent of the original Colab work—data acquisition/inspection, dataset creation, EDA, preprocessing, financial score, class labels, supervised model training, evaluation, comparison, saved model artifacts—while adapting it to the actual Excel columns and adding two comparable models beside Random Forest.

> **Important:** This notebook is a substantial, runnable **ML-stage minor project**, not a completed React + backend + Gemini web application. Weeks 6–10 include implementation-ready tasks and validation plans; those app components need their own code/repository.

### Before running
1. Upload this notebook and `Screener_Query_Results_Page1 (1).xlsx` to the same Google Colab session.
2. Run cells from top to bottom. If prompted, allow XGBoost installation.
3. Read each **What this cell does** note and write your own observations in the reflection cells.
4. Do not present generated-label agreement as proof of future stock returns or investment performance.

### Security note
The uploaded Python source contains a hard-coded Financial Modeling Prep API key. Do not reuse or publish that value. If it is a real key, revoke/rotate it. This Excel-based notebook does not need an API key. For a future live API workflow, use Colab Secrets or environment variables.

## Suggested 10–15 hour working schedule

These are **student working hours** for understanding, running, interpreting, documenting, and validating the project—not the time required for the notebook to execute.

| Project block | Planned hands-on time | Evidence to save |
|---|---:|---|
| Week 1 — problem definition, literature survey, workflow mapping | 1.0 h | objectives, scope, architecture notes |
| Week 2 — load workbook, rebuild repeated headers, audit data | 1.5 h | schema and missingness report |
| Week 3 — preprocessing and exploratory data analysis | 2.0 h | charts, outlier/correlation observations |
| Week 4 — feature engineering, score and class generation | 1.5 h | scoring explanation, class distribution |
| Week 5 — train and compare three models | 3.0 h | metrics, reports, confusion matrices, CV |
| Week 6 — model artifact / API / Gemini design | 0.75 h | inference contract and API plan |
| Week 7 — dashboard UX and frontend plan | 0.5 h | screen/module plan |
| Week 8 — integration and test design | 0.75 h | test checklist and test evidence |
| Week 9 — tuning, limitations and optimization | 0.75 h | error analysis and next experiment |
| Week 10 — report, slides and viva practice | 0.75 h | report outline, results narrative |
| **Total** | **12.5 h** | **A documented minor-project workflow** |

If you need to fit the project into 10 hours, shorten the optional extension exercises. For a 15-hour version, run the tuning and robustness exercises and write a fuller results discussion.

## Project objectives and research question

**Problem statement.** Financial indicators differ in scale and meaning. A learner may struggle to compare companies across valuation, profitability, growth and financial health. FinLearn demonstrates a transparent educational scoring workflow and compares three supervised classifiers.

**Objectives**
1. Read and audit the provided company-level financial workbook.
2. Standardize inconsistent repeated headers and safely convert numeric fields.
3. Inspect missingness, distributions, extreme values and correlations.
4. Create a directional financial score and three heuristic classes: `Undervalued`, `Fairly Valued`, `Overvalued`.
5. Train Random Forest, AdaBoost and XGBoost on the same split and features.
6. Compare accuracy, precision, recall, F1, confusion matrices, cross-validation and feature importance.
7. Save the fitted pipelines, class mapping, features, metrics and prediction outputs for later integration.

**Research question for this notebook:** Which of the three models most closely reproduces the rule-generated classes on a held-out sample of this workbook?

**Not the research question:** Can this dataset prove which stock will outperform in the future? No—there are no dated future returns in this workbook.

## Map from the original Colab file to this implementation

Your original script follows these stages. This notebook retains that order and purpose, then extends Week 5 from one model to three.

| Original Colab stage | How this notebook handles it | Reason for adaptation |
|---|---|---|
| Install/import `requests`, `pandas`, load FMP key | Import data-science libraries; use uploaded Excel | The supplied workbook is the runnable source; a live API key is not needed |
| Define US ticker list and inspect AAPL endpoints | Inspect workbook schema and sample rows | Workbook contains company names and Indian-market Screener indicators, not the US ticker API dataset |
| Fetch ratios, key metrics and growth for each ticker | Reconstruct the workbook's repeated headers and standardize available columns | Do not mix two different universes or fabricate unavailable metrics |
| Create `Financial_Features_Expanded.csv` | Create cleaned and standardized CSVs | Same intermediate-artifact concept |
| Week 2/3 missing values, numeric types, boxplots, histograms, correlation, pairplot, RobustScaler | Detailed data audit and EDA; demonstrate RobustScaler separately | Tree models do not require scaling; fit any learned preprocessing inside the training pipeline |
| Week 4 12-indicator score, ranks, 30th/70th percentile labels | Directional percentile ranks from eligible columns that actually exist | The Excel export lacks several original API fields such as Current Ratio, Revenue Growth, Net Income Growth, Net Profit Margin and PEG in consistent coverage |
| Week 5 Random Forest, 80/20 stratified split, metrics and saved model | Keep Random Forest baseline and add AdaBoost + XGBoost with the same split/metrics | Fair three-model comparison |

**Do not silently equate similarly named but different metrics.** For example, quarterly profit variation is not annual profit growth. The notebook keeps such fields distinct.

## Exact feature crosswalk: original API design vs supplied Excel

The original Colab used the following 12 feature names. The Excel source is a different dataset, so the mapping must be explicit:

| Original Colab feature | Excel field / status | Treatment in this notebook |
|---|---|---|
| `PE_Ratio` | `PE_Ratio` | Included if coverage/variation check passes |
| `PB_Ratio` | `Price_to_Book` (CMP/BV) | Included under the corresponding price-to-book field |
| `PEG_Ratio` | `PEG` | Included if coverage/variation check passes |
| `ROE` | `ROE_pct` | Included if coverage/variation check passes |
| `ROA` | `ROA_12M_pct` | Included if coverage/variation check passes |
| `Net_Profit_Margin` | No reliable equivalent in the workbook | Not fabricated |
| `Operating_Profit_Margin` | `OPM_pct`, but coverage is incomplete | Excluded if it falls below the documented coverage threshold |
| `Revenue_Growth` | No exact field | `Profit_Growth_pct` is not renamed to revenue growth | Not fabricated |
| `EPS_Growth` | `EPS_Var_5Yrs_pct` / `EPS_Var_3Yrs_pct` | Different time-window metrics; not treated as identical; excluded when coverage is too low |
| `Net_Income_Growth` | No reliable equivalent | Quarterly profit variation is not treated as the same measure | Not fabricated |
| `Debt_to_Equity` | `Debt_to_Equity` | Included if coverage/variation check passes |
| `Current_Ratio` | Not present | Not fabricated |

This is the core reason the new work cannot be a byte-for-byte reproduction of the API version: the source data and available features differ. The modelling stages and intent are preserved, while field mappings and scoring are adapted transparently.

## Week 1 — Scope, assumptions and methodology

### Methodology
1. **Data layer:** use the uploaded Screener workbook as a static snapshot.
2. **Preparation layer:** detect repeated header rows, map header variants to canonical names, convert numeric columns and report missingness.
3. **Scoring layer:** convert each eligible indicator to a percentile rank. Reverse ranks for indicators where lower values are treated as more favorable.
4. **Label layer:** define bottom 30% of scores as `Overvalued`, top 30% as `Undervalued`, and the middle 40% as `Fairly Valued`.
5. **ML layer:** fit three classifiers using a stratified 80/20 split and a median imputer inside each pipeline.
6. **Evaluation layer:** report multiple metrics, cross-validation and confusion matrices.
7. **Delivery layer:** export reproducible tables and model artifacts.

### Key assumption
The scoring direction is a **simplified educational heuristic**, not a universal finance law. Low P/E or low debt is not automatically good in every context; extreme/negative ratios may be misleading. Interpret scores as relative screening signals only.

## Week 2 — Load the Excel workbook

The workbook is not a perfectly rectangular table: it repeats headers and includes header-layout variants. We first read every row without assigning a single header, then detect rows that contain both the company identifier and serial-number field. Each company row is associated with the most recent header layout above it.

In [ ]:
# Cell 1 — Imports and reproducibility settings
# Run this first. The random seed makes train/test splitting and model randomness repeatable.
from pathlib import Path
import os
import re
import json
import warnings
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Working directory:", Path.cwd())

In [ ]:
# Cell 2 — Locate and read the workbook
# In Colab, upload the Excel file using the Files panel if it is not already present.
EXCEL_PATH = Path("Screener_Query_Results_Page1 (1).xlsx")

if not EXCEL_PATH.exists():
    try:
        from google.colab import files
        print("Please upload Screener_Query_Results_Page1 (1).xlsx")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")
        EXCEL_PATH = Path(next(iter(uploaded.keys())))
    except ImportError as exc:
        raise FileNotFoundError(
            "Workbook not found. Put the Excel workbook next to this notebook."
        ) from exc

raw = pd.read_excel(EXCEL_PATH, sheet_name=0, header=None, dtype=object)
print("Workbook:", EXCEL_PATH.name)
print("Raw workbook shape:", raw.shape)
display(raw.head(12))

In [ ]:
# Cell 3 — Normalize header text and detect repeated header rows
# A header row must include both a company field and a serial-number field.
def normalize_header(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower().replace("\n", " ")
    text = re.sub(r"[^a-z0-9%]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def looks_like_header(row):
    values = [normalize_header(v) for v in row.tolist()]
    has_company = "company" in values
    has_serial = any(v in {"s no", "sno", "serial no", "serial number"} for v in values)
    return has_company and has_serial

header_row_indices = [i for i, row in raw.iterrows() if looks_like_header(row)]
print("Detected repeated header rows:", len(header_row_indices))
print("First header row indices:", header_row_indices[:15])
if not header_row_indices:
    raise ValueError("Could not detect a row containing both S.No. and Company. Inspect raw.head().")

In [ ]:
# Cell 4 — Rebuild records under the correct header layout
# The latest detected header applies to subsequent data rows until another header appears.
records = []
current_headers = None

for row_index, row in raw.iterrows():
    if looks_like_header(row):
        current_headers = [str(v).strip() if not pd.isna(v) else "" for v in row.tolist()]
        continue
    if current_headers is None:
        continue
    row_values = row.tolist()
    if len(row_values) != len(current_headers):
        continue
    record = dict(zip(current_headers, row_values))
    company_value = next(
        (value for key, value in record.items() if normalize_header(key) == "company"),
        None
    )
    if pd.isna(company_value) or normalize_header(company_value) in {"", "company"}:
        continue
    record["_source_row"] = row_index
    records.append(record)

parsed_raw = pd.DataFrame(records)
print("Parsed rows before canonicalization:", len(parsed_raw))
print("Raw distinct header names:")
display(pd.Series(parsed_raw.columns, name="Header"))
display(parsed_raw.head(8))

In [ ]:
# Cell 5 — Standardize column names
# This maps observed header variants only. It does not create missing financial indicators.
def canonical_column(name):
    n = normalize_header(name)
    mapping = {
        "s no": "S_No", "sno": "S_No", "serial no": "S_No", "serial number": "S_No",
        "company": "Company",
        "cmp rs": "CMP_Rs", "cmp": "CMP_Rs",
        "p e": "PE_Ratio", "pe": "PE_Ratio", "p e ratio": "PE_Ratio",
        "cmp bv": "Price_to_Book", "price to book": "Price_to_Book", "pb": "Price_to_Book",
        "roe": "ROE_pct", "roe %": "ROE_pct",
        "roa 12m": "ROA_12M_pct", "roa 12m %": "ROA_12M_pct",
        "opm": "OPM_pct", "opm %": "OPM_pct",
        "profit growth": "Profit_Growth_pct", "profit growth %": "Profit_Growth_pct",
        "peg": "PEG",
        "eps 12m rs": "EPS_12M_Rs", "eps 12m": "EPS_12M_Rs",
        "eps var 5yrs": "EPS_Var_5Yrs_pct", "eps var 5yrs %": "EPS_Var_5Yrs_pct",
        "eps var 3yrs": "EPS_Var_3Yrs_pct", "eps var 3yrs %": "EPS_Var_3Yrs_pct",
        "debt eq": "Debt_to_Equity", "debt to equity": "Debt_to_Equity",
        "mar cap rs cr": "Market_Cap_Rs_Cr", "market cap rs cr": "Market_Cap_Rs_Cr",
        "roce": "ROCE_pct", "roce %": "ROCE_pct",
        "np qtr rs cr": "Net_Profit_Qtr_Rs_Cr", "net profit qtr rs cr": "Net_Profit_Qtr_Rs_Cr",
        "qtr profit var": "Quarterly_Profit_Var_pct", "qtr profit var %": "Quarterly_Profit_Var_pct",
    }
    return mapping.get(n, re.sub(r"[^A-Za-z0-9]+", "_", str(name)).strip("_"))

parsed = parsed_raw.copy()
parsed.columns = [canonical_column(c) for c in parsed.columns]

# If multiple original columns map to one canonical field, use the first non-missing value.
if parsed.columns.duplicated().any():
    merged = {}
    for col in dict.fromkeys(parsed.columns):
        same_named_columns = parsed.loc[:, parsed.columns == col]
        merged[col] = same_named_columns.bfill(axis=1).iloc[:, 0]
    parsed = pd.DataFrame(merged)

if "Company" not in parsed.columns:
    raise ValueError("Company column not found after canonicalization.")
parsed["Company"] = parsed["Company"].astype(str).str.strip()
parsed = parsed[parsed["Company"].notna() & ~parsed["Company"].isin(["", "nan", "None"])].copy()
for col in parsed.columns:
    if col not in {"Company"}:
        parsed[col] = pd.to_numeric(parsed[col], errors="coerce")

print("Canonicalized shape:", parsed.shape)
print("Canonical columns:", parsed.columns.tolist())
display(parsed.head(10))

In [ ]:
# Cell 6 — Duplicate handling and a first audit
# No timestamp exists in this workbook, so we cannot know which duplicate is newest.
# We retain the first row for a repeated company name and report the number removed.
duplicate_company_count = int(parsed["Company"].duplicated(keep="first").sum())
parsed = parsed.drop_duplicates(subset=["Company"], keep="first").reset_index(drop=True)

print("Duplicate company names removed:", duplicate_company_count)
print("Cleaned row count:", len(parsed))
print("Cleaned column count:", len(parsed.columns))
print("Duplicate full rows:", int(parsed.duplicated().sum()))
print("Company examples:")
display(parsed[["Company"]].head(10))
parsed.to_csv("Week2_Parsed_Screener_Data.csv", index=False)
print("Saved Week2_Parsed_Screener_Data.csv")

### Week 2 — Data dictionary and coverage audit

This audit is important because the original FMP workflow expected 12 features: PE, PB, PEG, ROE, ROA, net profit margin, operating profit margin, revenue growth, EPS growth, net-income growth, debt-to-equity and current ratio. The Excel workbook does **not** provide all of those in consistent coverage. We must not fabricate them or relabel a different measure as if it were identical.

In [ ]:
# Cell 7 — Column-level coverage and types
schema_report = pd.DataFrame({
    "Column": parsed.columns,
    "Data_Type": [str(parsed[c].dtype) for c in parsed.columns],
    "Non_Missing": [int(parsed[c].notna().sum()) for c in parsed.columns],
    "Missing": [int(parsed[c].isna().sum()) for c in parsed.columns],
    "Missing_Percent": [(parsed[c].isna().mean() * 100).round(2) for c in parsed.columns],
    "Unique_Non_Missing": [int(parsed[c].nunique(dropna=True)) for c in parsed.columns],
})
display(schema_report)
schema_report.to_csv("Week2_Schema_Coverage_Report.csv", index=False)

print("Potential feature columns available:")
print([c for c in parsed.columns if c not in {"Company", "_source_row", "S_No"}])

In [ ]:
# Cell 8 — Numeric descriptive statistics
numeric_columns = parsed.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", len(numeric_columns))
display(parsed[numeric_columns].describe().T)
print("\nMissing values by column:")
display(parsed.isna().sum().sort_values(ascending=False).to_frame("Missing_Count"))

# Save a stable intermediate dataset for repeatable work.
parsed.to_csv("Week2_Parsed_Screener_Data.csv", index=False)

## Week 3 — Data preprocessing and exploratory data analysis

The original notebook checked infinite values, missing values, duplicates, numerical distributions, boxplots, correlations, high correlations, pairwise relationships and a RobustScaler version. This section keeps those analytical steps but avoids destructive preprocessing. Outliers are flagged for investigation rather than automatically deleted: a very high P/E or debt ratio may be real, though it may also need special interpretation.

In [ ]:
# Cell 9 — Replace infinite values with missing values and inspect missingness
numeric_columns = parsed.select_dtypes(include=np.number).columns.tolist()
if numeric_columns:
    numeric_array = parsed[numeric_columns].to_numpy(dtype=float)
    infinite_report = pd.Series(np.isinf(numeric_array).sum(axis=0), index=numeric_columns, name="Infinite_Count")
    display(infinite_report.to_frame())
    parsed[numeric_columns] = parsed[numeric_columns].replace([np.inf, -np.inf], np.nan)

missing_report = pd.DataFrame({
    "Missing_Count": parsed.isna().sum(),
    "Missing_Percent": (parsed.isna().mean() * 100).round(2),
}).sort_values("Missing_Percent", ascending=False)
display(missing_report)
missing_report.to_csv("Week3_Missingness_Report.csv")

In [ ]:
# Cell 10 — Outlier summary using the 1.5 × IQR rule
# IQR is a flagging rule, not a command to delete a row.
outlier_rows = []
for col in numeric_columns:
    series = parsed[col].dropna()
    if series.empty:
        continue
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({
        "Feature": col,
        "Valid_Count": len(series),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(((series < lower) | (series > upper)).sum()),
    })
outlier_report = pd.DataFrame(outlier_rows).sort_values("Outlier_Count", ascending=False)
display(outlier_report)
outlier_report.to_csv("Week3_IQR_Outlier_Report.csv", index=False)

In [ ]:
# Cell 11 — Histograms for the numeric features
# Limit the first display to 12 columns so the figure remains readable.
plot_columns = [c for c in numeric_columns if parsed[c].notna().sum() > 0][:12]
if plot_columns:
    parsed[plot_columns].hist(figsize=(16, 12), bins=25, edgecolor="black")
    plt.suptitle("Week 3 — Distributions of Selected Financial Features", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns available to plot.")

In [ ]:
# Cell 12 — Boxplots to visually inspect extremes
for col in plot_columns:
    plt.figure(figsize=(8, 2.8))
    sns.boxplot(x=parsed[col], color="steelblue")
    plt.title(f"Boxplot: {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 13 — Correlation matrix
# Correlation is meaningful only for numeric columns and does not prove causation.
correlation_columns = [c for c in numeric_columns if parsed[c].notna().sum() >= 3 and parsed[c].nunique(dropna=True) > 1]
if len(correlation_columns) >= 2:
    corr = parsed[correlation_columns].corr(method="pearson")
    plt.figure(figsize=(min(16, 1 + 0.8 * len(correlation_columns)), min(12, 1 + 0.7 * len(correlation_columns))))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=len(correlation_columns) <= 12, fmt=".2f")
    plt.title("Week 3 — Correlation Matrix")
    plt.tight_layout()
    plt.show()
    corr.to_csv("Week3_Correlation_Matrix.csv")
else:
    print("Not enough varying numeric columns for a correlation matrix.")

In [ ]:
# Cell 14 — Find strongly correlated pairs (absolute correlation >= 0.80)
strong_pairs = []
if len(correlation_columns) >= 2:
    corr = parsed[correlation_columns].corr()
    for i, left in enumerate(correlation_columns):
        for right in correlation_columns[i + 1:]:
            value = corr.loc[left, right]
            if pd.notna(value) and abs(value) >= 0.80:
                strong_pairs.append({"Feature_1": left, "Feature_2": right, "Correlation": value})
strong_corr_report = pd.DataFrame(strong_pairs, columns=["Feature_1", "Feature_2", "Correlation"])
print("Highly correlated pairs:", len(strong_corr_report))
display(strong_corr_report)
strong_corr_report.to_csv("Week3_High_Correlation_Pairs.csv", index=False)

In [ ]:
# Cell 15 — Pairwise plots for a small set of important indicators
# pandas scatter_matrix is used here for broad compatibility across seaborn/matplotlib versions.
from pandas.plotting import scatter_matrix
preferred_pair_features = [
    "PE_Ratio", "Price_to_Book", "PEG", "Debt_to_Equity",
    "ROE_pct", "Profit_Growth_pct", "EPS_Var_5Yrs_pct", "ROCE_pct"
]
pair_features = [c for c in preferred_pair_features if c in parsed.columns and parsed[c].notna().sum() > 3]
if len(pair_features) >= 2:
    pair_sample = parsed[pair_features].dropna(how="all").sample(
        n=min(400, len(parsed[pair_features].dropna(how="all"))), random_state=RANDOM_STATE
    )
    axes = scatter_matrix(pair_sample, figsize=(14, 14), diagonal="hist", alpha=0.35)
    plt.suptitle("Week 3 — Pairwise Feature Relationships (sample)", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough selected indicators for pairwise plotting.")

In [ ]:
# Cell 16 — Demonstrate RobustScaler, mirroring the original Colab exploration
# Important: this is an EDA demonstration only. Do not fit a scaler on the full dataset
# and then evaluate a model on a test split. In this notebook, the model pipelines impute
# values within each training fold; tree models do not require feature scaling.
from sklearn.preprocessing import RobustScaler

scale_candidates = [c for c in correlation_columns if parsed[c].notna().sum() >= 3]
if scale_candidates:
    scaler_demo_data = parsed[scale_candidates].copy()
    scaler_demo_data = scaler_demo_data.fillna(scaler_demo_data.median(numeric_only=True))
    scaler_demo = RobustScaler()
    scaled_values = scaler_demo.fit_transform(scaler_demo_data)
    scaled_demo = pd.DataFrame(scaled_values, columns=scale_candidates, index=parsed.index)
    scaling_comparison = pd.DataFrame({
        "Original_Median": scaler_demo_data.median(),
        "Original_IQR": scaler_demo_data.quantile(.75) - scaler_demo_data.quantile(.25),
        "Scaled_Median": scaled_demo.median(),
        "Scaled_IQR": scaled_demo.quantile(.75) - scaled_demo.quantile(.25),
    })
    display(scaling_comparison)
    scaled_demo.to_csv("Week3_RobustScaled_EDA_Demo.csv", index=False)
else:
    print("No numeric columns available for RobustScaler demonstration.")

### Week 3 — Preprocessing decisions

- Missing numeric values remain missing during EDA; they are not automatically replaced with zero.
- A model feature must have enough observations and meaningful variation.
- The three models use a `Pipeline` with `SimpleImputer(strategy="median")`, so the imputer is fitted on the training portion, not the entire dataset.
- We do not drop all outliers automatically.
- `RobustScaler` is demonstrated to mirror your old work, but is not used in the tree-model pipelines because Random Forest, AdaBoost and XGBoost do not generally require scaled inputs.

**Checkpoint:** write 3–5 observations after inspecting the reports. For example, identify the most-missing feature, one skewed feature and one strong correlation. Use the actual outputs rather than guessing.

In [ ]:
# Cell 17 — Your Week 3 observations (fill these in after looking at the charts)
week3_observations = {
    "Most_missing_feature": "Fill in from Week3_Missingness_Report.csv",
    "Most_skewed_or_extreme_feature": "Fill in after inspecting histograms/boxplots",
    "One_strong_correlation": "Fill in from Week3_High_Correlation_Pairs.csv",
    "Preprocessing_decision": "Explain why missing values are imputed inside the model pipeline",
}
print(json.dumps(week3_observations, indent=2))

## Week 4 — Feature engineering and rule-based label generation

### Important adaptation from the original Colab

The original FMP dataset expected 12 features, including Current Ratio, Revenue Growth, Net Income Growth, Net Profit Margin and PEG. The supplied workbook does not have all of them with reliable coverage. The following code audits candidate fields and uses only available columns with sufficient non-missing observations and variation.

**Direction convention for this simplified score**
- Lower is treated as more favorable: P/E, price-to-book, PEG, debt-to-equity.
- Higher is treated as more favorable: ROE, ROA, operating margin, profit growth, EPS variation, ROCE.
- These directions are heuristic. A negative P/E or extreme ratio can require separate interpretation.

Instead of averaging raw financial ratios with incompatible scales, each selected metric is converted into a percentile rank. This is an intentional methodological improvement over averaging raw or scaled values with potentially different ranges.

In [ ]:
# Cell 18 — Define candidate features and score directions
feature_direction = {
    "PE_Ratio": -1,
    "Price_to_Book": -1,
    "PEG": -1,
    "ROE_pct": 1,
    "ROA_12M_pct": 1,
    "OPM_pct": 1,
    "Profit_Growth_pct": 1,
    "EPS_Var_5Yrs_pct": 1,
    "EPS_Var_3Yrs_pct": 1,
    "Debt_to_Equity": -1,
    "ROCE_pct": 1,
}

feature_audit_rows = []
for feature, direction in feature_direction.items():
    if feature not in parsed.columns:
        feature_audit_rows.append({
            "Feature": feature, "Available_In_Workbook": False,
            "Non_Missing": 0, "Missing_Percent": 100.0,
            "Unique_Values": 0, "Direction": direction, "Eligible": False,
            "Reason": "Column not present in this Excel export",
        })
        continue
    valid_count = int(parsed[feature].notna().sum())
    missing_pct = float(parsed[feature].isna().mean() * 100)
    unique_count = int(parsed[feature].nunique(dropna=True))
    eligible = valid_count >= max(30, int(0.50 * len(parsed))) and unique_count > 1
    feature_audit_rows.append({
        "Feature": feature, "Available_In_Workbook": True,
        "Non_Missing": valid_count, "Missing_Percent": round(missing_pct, 2),
        "Unique_Values": unique_count, "Direction": direction, "Eligible": eligible,
        "Reason": "Eligible" if eligible else "Insufficient coverage or variation",
    })
feature_audit = pd.DataFrame(feature_audit_rows)
display(feature_audit)
feature_audit.to_csv("Week4_Feature_Audit.csv", index=False)

score_features = feature_audit.loc[feature_audit["Eligible"], "Feature"].tolist()
if len(score_features) < 3:
    raise ValueError("Fewer than 3 eligible financial features. Review Week4_Feature_Audit.csv.")
print("Selected scoring features:", score_features)

In [ ]:
# Cell 19 — Convert financial features into directional percentile ranks
# A percentile rank is between 0 and 1. Higher means more favorable after direction adjustment.
rank_features = pd.DataFrame(index=parsed.index)
for feature in score_features:
    percentile_rank = parsed[feature].rank(pct=True, method="average")
    if feature_direction[feature] < 0:
        percentile_rank = 1.0 - percentile_rank
    rank_features[feature + "_DirectionalRank"] = percentile_rank

rank_features["Financial_Score"] = rank_features.mean(axis=1, skipna=True)
rank_features["Score_Feature_Count"] = rank_features.notna().sum(axis=1)
parsed["Financial_Score"] = rank_features["Financial_Score"]
parsed["Score_Feature_Count"] = rank_features["Score_Feature_Count"]

print("Directional rank preview:")
display(rank_features.head(10))
print("Financial score summary:")
display(parsed["Financial_Score"].describe().to_frame())

In [ ]:
# Cell 20 — Generate three heuristic labels using the 30th and 70th percentiles
# The labels are relative to this workbook snapshot, not a verified market valuation.
model_base = parsed.loc[parsed["Score_Feature_Count"] >= 3].copy()
model_base = model_base[model_base["Financial_Score"].notna()].reset_index(drop=True)

low_threshold = float(model_base["Financial_Score"].quantile(0.30))
high_threshold = float(model_base["Financial_Score"].quantile(0.70))

def assign_valuation_label(score):
    if score <= low_threshold:
        return "Overvalued"
    if score >= high_threshold:
        return "Undervalued"
    return "Fairly Valued"

model_base["Label"] = model_base["Financial_Score"].apply(assign_valuation_label)
model_base["Financial_Rank"] = model_base["Financial_Score"].rank(ascending=False, method="min").astype(int)
model_base = model_base.sort_values("Financial_Score", ascending=False).reset_index(drop=True)

print(f"30th-percentile threshold: {low_threshold:.5f}")
print(f"70th-percentile threshold: {high_threshold:.5f}")
print("Rows available for modeling:", len(model_base))
display(model_base[["Company", "Financial_Score", "Financial_Rank", "Score_Feature_Count", "Label"]].head(20))
print("Class distribution:")
display(model_base["Label"].value_counts().to_frame("Count"))
model_base.to_csv("Week4_Financial_Data_With_Scores.csv", index=False)

In [ ]:
# Cell 21 — Visualize score and class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.countplot(data=model_base, x="Label", order=["Undervalued", "Fairly Valued", "Overvalued"], ax=axes[0], color="steelblue")
axes[0].set_title("Rule-generated valuation classes")
axes[0].set_xlabel("Heuristic class")
axes[0].set_ylabel("Company count")

sns.histplot(model_base["Financial_Score"], bins=25, kde=True, ax=axes[1], color="darkorange")
axes[1].axvline(low_threshold, linestyle="--", label="30th percentile")
axes[1].axvline(high_threshold, linestyle="--", label="70th percentile")
axes[1].set_title("Financial score distribution")
axes[1].set_xlabel("Mean directional percentile rank")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 22 — Inspect the strongest and weakest heuristic scores
columns_to_show = ["Company", "Financial_Score", "Financial_Rank", "Label"] + score_features
print("Top 15 heuristic scores:")
display(model_base[columns_to_show].head(15))
print("Bottom 15 heuristic scores:")
display(model_base[columns_to_show].tail(15))
model_base[columns_to_show].to_csv("Week4_Ranked_Company_Scores.csv", index=False)

### Week 4 — How to explain the score in a viva

For each company, the code ranks each selected indicator across companies. A metric where higher is considered favorable keeps its percentile rank; a metric where lower is considered favorable uses `1 - percentile_rank`. The financial score is the mean of available directional ranks. Then:

- score at or below the 30th percentile → `Overvalued` (heuristic label)
- score at or above the 70th percentile → `Undervalued` (heuristic label)
- score between those thresholds → `Fairly Valued`

The naming is a project convention. A high composite score does not prove intrinsic value is below market price. To make a genuine valuation claim, you would need a defensible valuation methodology and/or future-outcome validation.

In [ ]:
# Cell 23 — Prepare the ML dataset using only real, eligible features
# Keep identifiers and score out of X. Financial_Score and Label are target-generation outputs.
model_features = score_features.copy()
X = model_base[model_features].replace([np.inf, -np.inf], np.nan).copy()
y_text = model_base["Label"].astype(str).copy()
company_ids = model_base["Company"].astype(str).copy()
financial_scores = model_base["Financial_Score"].copy()

# Exclude records where every selected model feature is missing.
valid_rows = X.notna().any(axis=1)
X = X.loc[valid_rows].reset_index(drop=True)
y_text = y_text.loc[valid_rows].reset_index(drop=True)
company_ids = company_ids.loc[valid_rows].reset_index(drop=True)
financial_scores = financial_scores.loc[valid_rows].reset_index(drop=True)

print("X shape:", X.shape)
print("Feature columns:", model_features)
print("Target class counts:")
display(y_text.value_counts().to_frame("Count"))
print("Any all-missing feature rows left?", bool(X.isna().all(axis=1).any()))

if y_text.nunique() < 2:
    raise ValueError("Classification needs at least two target classes.")
if int(y_text.value_counts().min()) < 2:
    raise ValueError("Each class needs at least two examples for stratified splitting.")

## Week 5 — Train and compare three machine-learning models

### Experimental design
- Same cleaned records, feature columns and stratified 80/20 split for all models.
- `random_state=42` for reproducibility.
- Median imputation occurs inside each pipeline to prevent the imputer from learning test-set medians.
- Label encoding is applied to the three text classes because XGBoost versions can require integer class labels.
- The test set is used for the final holdout comparison; cross-validation is a second stability check.

**Critical interpretation:** labels are derived from the same financial features supplied to the classifiers. These experiments measure how closely the models approximate the rule-generated labels. They do not establish independent future-price prediction.

In [ ]:
# Cell 24 — Encode labels and create a stratified train/test split
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    balanced_accuracy_score
)
import sklearn
import joblib

from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = label_encoder.classes_.tolist()
print("scikit-learn version:", sklearn.__version__)
print("Class-to-integer mapping:", {name: int(label_encoder.transform([name])[0]) for name in class_names})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training class counts:")
display(pd.Series(label_encoder.inverse_transform(y_train)).value_counts().to_frame("Count"))
print("Testing class counts:")
display(pd.Series(label_encoder.inverse_transform(y_test)).value_counts().to_frame("Count"))

In [ ]:
# Cell 25 — Install/import XGBoost
# Colab usually supports this pip magic. If XGBoost is already installed, it will be updated only if needed.
%pip -q install xgboost

from xgboost import XGBClassifier
import xgboost
print("XGBoost version:", xgboost.__version__)

In [ ]:
# Cell 26 — Define the three model pipelines
# The imputer is part of each pipeline so each fit learns medians from its training data only.
models = {
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
            min_samples_leaf=2,
        )),
    ]),
    "AdaBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            n_estimators=200,
            learning_rate=0.5,
            random_state=RANDOM_STATE,
        )),
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
        )),
    ]),
}
print("Models defined:", list(models.keys()))

### Model 1 — Random Forest

Random Forest trains many decision trees using randomized samples/features, then combines their votes. It is a strong baseline for tabular financial data and can model nonlinear relationships. In this experiment, 300 trees are used; `class_weight="balanced"` adjusts for differences in class frequency. It is not inherently an explanation of why a company is truly undervalued.

In [ ]:
# Cell 27 — Train and evaluate Random Forest first (baseline)
rf_model = models["Random Forest"]
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print("Random Forest training complete.")
print("Holdout accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print(classification_report(y_test, rf_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0))

### Model 2 — AdaBoost

AdaBoost trains weak learners sequentially. Examples that are misclassified receive more attention in later rounds. The default weak learner here is a decision stump (`max_depth=1`). The learning rate controls how strongly each learner contributes. AdaBoost can be sensitive to noisy labels, which matters because our labels come from a heuristic rule.

In [ ]:
# Cell 28 — Train and evaluate AdaBoost
ada_model = models["AdaBoost"]
ada_model.fit(X_train, y_train)
ada_pred = ada_model.predict(X_test)
print("AdaBoost training complete.")
print("Holdout accuracy:", round(accuracy_score(y_test, ada_pred), 4))
print(classification_report(y_test, ada_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0))

### Model 3 — XGBoost

XGBoost builds trees in sequence, with each new tree helping reduce the loss left by previous trees. It includes regularization and efficient training options. The learning rate is 0.05, maximum tree depth is 3, and row/column subsampling is used to reduce overfitting. These are starting parameters, not guaranteed optimal settings.

In [ ]:
# Cell 29 — Train and evaluate XGBoost
xgb_model = models["XGBoost"]
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
print("XGBoost training complete.")
print("Holdout accuracy:", round(accuracy_score(y_test, xgb_pred), 4))
print(classification_report(y_test, xgb_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0))

### Common comparison — same test set and same metrics

- **Accuracy:** overall proportion of correct predictions.
- **Precision (weighted):** when the model predicts a class, how often it is correct, aggregated with class-frequency weights.
- **Recall (weighted):** fraction of actual examples recovered, aggregated with class-frequency weights.
- **F1 (weighted):** weighted harmonic mean of precision and recall.
- **Macro F1:** calculates F1 for each class and averages them equally; useful when class balance is important.
- **Balanced accuracy:** average recall across classes.

Do not choose a model based only on accuracy. Review class-wise recall and macro F1 as well.

In [ ]:
# Cell 30 — Evaluate all three models using a common function
predictions = {
    "Random Forest": rf_pred,
    "AdaBoost": ada_pred,
    "XGBoost": xgb_pred,
}
fitted_models = {
    "Random Forest": rf_model,
    "AdaBoost": ada_model,
    "XGBoost": xgb_model,
}

metric_rows = []
for model_name, pred in predictions.items():
    metric_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_test, pred),
        "Precision_Weighted": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall_Weighted": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1_Weighted": f1_score(y_test, pred, average="weighted", zero_division=0),
        "Precision_Macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall_Macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1_Macro": f1_score(y_test, pred, average="macro", zero_division=0),
    })
metrics_df = pd.DataFrame(metric_rows).sort_values("F1_Macro", ascending=False).reset_index(drop=True)
display(metrics_df.style.format({col: "{:.3f}" for col in metrics_df.columns if col != "Model"}))
metrics_df.to_csv("Week5_Model_Comparison_Test_Metrics.csv", index=False)
print("Saved Week5_Model_Comparison_Test_Metrics.csv")

In [ ]:
# Cell 31 — Plot the metric comparison
metric_plot_columns = ["Accuracy", "F1_Weighted", "F1_Macro", "Balanced_Accuracy"]
plot_metrics = metrics_df.set_index("Model")[metric_plot_columns]
plot_metrics.plot(kind="bar", figsize=(11, 5), ylim=(0, 1), rot=0)
plt.title("Week 5 — Three-Model Holdout Comparison")
plt.ylabel("Score (0 to 1)")
plt.xlabel("Model")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 32 — Detailed class-wise reports and confusion matrices
for model_name, pred in predictions.items():
    print("\n" + "=" * 80)
    print(model_name, "— classification report")
    print(classification_report(
        y_test, pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        digits=3,
        zero_division=0,
    ))
    cm = confusion_matrix(y_test, pred, labels=np.arange(len(class_names)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(values_format="d", xticks_rotation=25)
    plt.title(f"{model_name} — Confusion Matrix")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 33 — Stratified cross-validation for stability
# Cross-validation scores still measure reproduction of heuristic labels.
from sklearn.model_selection import StratifiedKFold, cross_validate
minimum_class_count = int(pd.Series(y).value_counts().min())
n_splits = min(5, minimum_class_count)
cv_rows = []

if n_splits >= 2:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        "accuracy": "accuracy",
        "precision_weighted": "precision_weighted",
        "recall_weighted": "recall_weighted",
        "f1_weighted": "f1_weighted",
        "f1_macro": "f1_macro",
    }
    for model_name, pipeline in models.items():
        cv_result = cross_validate(
            pipeline, X, y, cv=cv, scoring=scoring,
            n_jobs=1, return_train_score=False, error_score="raise"
        )
        row = {"Model": model_name, "CV_Folds": n_splits}
        for metric_name in scoring:
            values = cv_result["test_" + metric_name]
            row[metric_name + "_Mean"] = float(values.mean())
            row[metric_name + "_Std"] = float(values.std())
        cv_rows.append(row)
    cv_df = pd.DataFrame(cv_rows).sort_values("f1_macro_Mean", ascending=False).reset_index(drop=True)
    display(cv_df.style.format({col: "{:.3f}" for col in cv_df.columns if col not in {"Model", "CV_Folds"}}))
    cv_df.to_csv("Week5_Model_Comparison_Cross_Validation.csv", index=False)
    print("Saved Week5_Model_Comparison_Cross_Validation.csv")
else:
    cv_df = pd.DataFrame()
    print("Cross-validation skipped because a class has fewer than two records.")

### Week 5 — Feature importance and error analysis

Tree feature importance indicates which inputs the fitted model used most often/usefully according to its importance measure. It is **not causality** and can be unstable when features are correlated. The test prediction table helps identify which companies were classified differently from the rule-generated label.

In [ ]:
# Cell 34 — Feature importance for the three tree-based models
importance_frames = []
for model_name, pipeline in fitted_models.items():
    estimator = pipeline.named_steps["classifier"]
    if hasattr(estimator, "feature_importances_"):
        importance_frames.append(pd.DataFrame({
            "Model": model_name,
            "Feature": model_features,
            "Importance": estimator.feature_importances_,
        }))

if importance_frames:
    importance_df = pd.concat(importance_frames, ignore_index=True)
    display(importance_df.sort_values(["Model", "Importance"], ascending=[True, False]))
    importance_df.to_csv("Week5_Feature_Importance.csv", index=False)
    for model_name in importance_df["Model"].unique():
        subset = importance_df[importance_df["Model"] == model_name].sort_values("Importance")
        plt.figure(figsize=(8, max(4, 0.35 * len(subset))))
        plt.barh(subset["Feature"], subset["Importance"])
        plt.title(f"{model_name} — Feature Importance")
        plt.xlabel("Model-specific importance")
        plt.tight_layout()
        plt.show()
else:
    importance_df = pd.DataFrame()
    print("No feature importance was exposed by these estimators.")

In [ ]:
# Cell 35 — Build a row-level test prediction table
# This is an error-analysis artifact, not a recommendation list.
test_indices = X_test.index.to_numpy()
prediction_table = pd.DataFrame({
    "Company": company_ids.iloc[test_indices].to_numpy(),
    "Financial_Score": financial_scores.iloc[test_indices].to_numpy(),
    "Rule_Label": label_encoder.inverse_transform(y_test),
})
for model_name, pred in predictions.items():
    safe_name = model_name.replace(" ", "_")
    prediction_table[safe_name + "_Prediction"] = label_encoder.inverse_transform(pred)
    prediction_table[safe_name + "_Agrees_With_Rule"] = (
        prediction_table["Rule_Label"] == prediction_table[safe_name + "_Prediction"]
    )
display(prediction_table.head(30))
prediction_table.to_csv("Week5_Test_Set_Predictions.csv", index=False)
print("Saved Week5_Test_Set_Predictions.csv")

In [ ]:
# Cell 36 — Review the disagreement cases
# Inspect these cases to understand what patterns the model finds difficult to reproduce.
model_agreement_columns = [c for c in prediction_table.columns if c.endswith("_Agrees_With_Rule")]
for column in model_agreement_columns:
    print(column, "agreement rate:", round(prediction_table[column].mean() * 100, 2), "%")

if model_agreement_columns:
    any_disagreement = ~prediction_table[model_agreement_columns].all(axis=1)
    print("Rows where at least one model disagrees with the rule:", int(any_disagreement.sum()))
    display(prediction_table.loc[any_disagreement].head(25))

### Interpretation worksheet — complete after running Week 5

Use the actual metric tables and plots. Do not copy made-up performance numbers into your report.

In [ ]:
# Cell 37 — Fill in your model comparison narrative after reviewing metrics_df and cv_df
results_interpretation = {
    "Best_holdout_model_by_macro_F1": "Fill in from Week5_Model_Comparison_Test_Metrics.csv",
    "Best_cross_validation_model_by_macro_F1": "Fill in from Week5_Model_Comparison_Cross_Validation.csv",
    "Class_with_lowest_recall": "Inspect each classification report",
    "Most_important_features": "Inspect Week5_Feature_Importance.csv",
    "Does_holdout_and_CV_ranking_agree": "Compare the two tables",
    "Why_results_do_not_prove_future_returns": "Labels were generated from a score based on these features",
}
print(json.dumps(results_interpretation, indent=2))

## Week 5 — Refit final pipelines and save artifacts

After evaluation, the pipelines are refitted on all available rows so the artifacts can be used for a demonstration. **Do not evaluate these refitted models on the same rows and report that as test performance.** Use the previously saved holdout metrics for the comparison.

In [ ]:
# Cell 38 — Refit final versions and export reproducible artifacts
final_models = {}
for model_name, pipeline in models.items():
    pipeline.fit(X, y)
    final_models[model_name] = pipeline
    print("Refitted on all modeling rows:", model_name)

ARTIFACT_DIR = Path("finlearn_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
for model_name, pipeline in final_models.items():
    filename = model_name.lower().replace(" ", "_") + "_pipeline.joblib"
    joblib.dump(pipeline, ARTIFACT_DIR / filename)
    print("Saved:", ARTIFACT_DIR / filename)

joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")
with open(ARTIFACT_DIR / "model_features.json", "w", encoding="utf-8") as f:
    json.dump(model_features, f, indent=2)
with open(ARTIFACT_DIR / "label_classes.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)
metrics_df.to_json(ARTIFACT_DIR / "test_metrics.json", orient="records", indent=2)
if not cv_df.empty:
    cv_df.to_json(ARTIFACT_DIR / "cross_validation_metrics.json", orient="records", indent=2)
if not importance_df.empty:
    importance_df.to_csv(ARTIFACT_DIR / "feature_importance.csv", index=False)
print("Artifact directory contents:", sorted(p.name for p in ARTIFACT_DIR.iterdir()))

In [ ]:
# Cell 39 — Demonstrate loading a saved pipeline and predicting one company row
# The input must contain exactly the model features used during training.
example_input = X.iloc[[0]].copy()
example_company = company_ids.iloc[0]
print("Example company:", example_company)
display(example_input)

loaded_rf = joblib.load(ARTIFACT_DIR / "random_forest_pipeline.joblib")
loaded_encoder = joblib.load(ARTIFACT_DIR / "label_encoder.joblib")
example_prediction_encoded = loaded_rf.predict(example_input)
example_prediction = loaded_encoder.inverse_transform(example_prediction_encoded)[0]
print("Random Forest demo prediction:", example_prediction)
print("Reminder: this is a prediction of the heuristic class, not a guaranteed investment signal.")

In [ ]:
# Cell 40 — Export all three model predictions for the entire current snapshot
# This is useful for a dashboard prototype. It is not a holdout evaluation.
full_prediction_export = model_base[["Company", "Financial_Score", "Financial_Rank", "Label"]].copy()
# Align with the model rows, which exclude records where all chosen model features are missing.
full_prediction_export = full_prediction_export.loc[valid_rows.to_numpy()].reset_index(drop=True)
for model_name, pipeline in final_models.items():
    pred_codes = pipeline.predict(X)
    full_prediction_export[model_name.replace(" ", "_") + "_Prediction"] = label_encoder.inverse_transform(pred_codes)
full_prediction_export.to_csv("Week5_All_Companies_Model_Predictions.csv", index=False)
print("Saved Week5_All_Companies_Model_Predictions.csv")
display(full_prediction_export.head(20))

## Week 6 — Backend API and Gemini explanation design

This notebook saves the ML artifacts, but it does **not** create a deployed backend or call Gemini. The next application stage should expose a prediction endpoint and keep the model's structured output separate from the generated explanation.

### Suggested API contract
- `POST /api/evaluate`: accept a company identifier and validated financial feature values.
- Validate that every required feature is present or explicitly missing; reject non-numeric strings.
- Load the saved pipeline and label encoder; return `predicted_class` and, if supported, class probabilities.
- Return a separate `score_method` and `data_timestamp` field where known.
- Gemini explanation should only explain supplied metrics and model output; it must not invent missing values, future prices or guaranteed returns.

**Security:** keep API keys server-side in environment variables/secret storage. Never expose a Gemini or financial-data API key in frontend JavaScript or a public notebook.

In [ ]:
# Cell 41 — Create a minimal inference helper for later API integration
# This is a notebook helper, not a deployed HTTP server.
def predict_company_class(feature_record, model_name="Random Forest"):
    """Predict the heuristic class for one validated feature record."""
    if model_name not in final_models:
        raise ValueError(f"Unknown model_name. Choose one of: {list(final_models)}")
    missing_features = [feature for feature in model_features if feature not in feature_record]
    if missing_features:
        raise ValueError(f"Missing required feature keys: {missing_features}")
    row = pd.DataFrame([{feature: feature_record[feature] for feature in model_features}])
    for feature in model_features:
        row[feature] = pd.to_numeric(row[feature], errors="coerce")
    pred_code = final_models[model_name].predict(row)
    predicted_class = label_encoder.inverse_transform(pred_code)[0]
    result = {"model": model_name, "predicted_class": predicted_class,
              "label_meaning": "rule-generated educational category; not an investment recommendation"}
    if hasattr(final_models[model_name].named_steps["classifier"], "predict_proba"):
        probabilities = final_models[model_name].predict_proba(row)[0]
        result["class_probabilities"] = {
            class_name: float(probability)
            for class_name, probability in zip(class_names, probabilities)
        }
    return result

example_record = X.iloc[0].to_dict()
print(predict_company_class(example_record, "Random Forest"))

## Week 7 — Frontend plan (React + TypeScript)

Build these modules after the backend contract is stable:

1. **Stock dashboard:** search company, display financial indicators, score and class with a clear educational disclaimer.
2. **Model comparison:** show Random Forest, AdaBoost and XGBoost predictions and evaluation metrics; distinguish per-company predictions from test-set performance.
3. **Company comparison:** compare two or more companies on the same available indicators and clearly mark missing fields.
4. **Portfolio analysis:** show the user's selected companies and summarized exposures; do not imply guaranteed returns.
5. **AI learning assistant:** explain terms like P/E, ROE, debt-to-equity and why the models differ. Ground explanations in supplied data.

Frontend should never contain secret API keys. A React app should call the backend rather than loading `.joblib` files directly.

## Week 8 — Integration and testing checklist

Mark each test only after you actually perform it.

In [ ]:
# Cell 42 — Notebook-level validation checks
validation_results = []
def record_check(name, passed, detail=""):
    validation_results.append({"Check": name, "Passed": bool(passed), "Detail": str(detail)})

record_check("Company column exists", "Company" in parsed.columns, parsed.shape)
record_check("Company identifiers are non-empty", parsed["Company"].notna().all(), "Parsed workbook")
record_check("Selected features exist", all(c in X.columns for c in model_features), model_features)
record_check("Target has at least two classes", len(np.unique(y)) >= 2, class_names)
record_check("No all-missing rows in X", not X.isna().all(axis=1).any(), X.shape)
record_check("All three models fitted", set(final_models) == {"Random Forest", "AdaBoost", "XGBoost"}, list(final_models))
record_check("Saved pipeline files exist", all((ARTIFACT_DIR / (n.lower().replace(" ", "_") + "_pipeline.joblib")).exists() for n in final_models), ARTIFACT_DIR)
validation_df = pd.DataFrame(validation_results)
display(validation_df)
validation_df.to_csv("Week8_Notebook_Validation_Checks.csv", index=False)
if not validation_df["Passed"].all():
    warnings.warn("One or more validation checks failed. Review Week8_Notebook_Validation_Checks.csv.")

### Additional integration tests to perform when the app exists

- [ ] API rejects missing required fields and non-numeric feature values.
- [ ] Saved model loads in a fresh Python process.
- [ ] API returns the same class names as `label_encoder.joblib`.
- [ ] Frontend shows unavailable indicators as unavailable, not zero.
- [ ] Gemini explanation uses only the structured feature values supplied to it.
- [ ] Error messages do not reveal API keys or internal paths.
- [ ] End-to-end flow works: company input → feature validation → model prediction → explanation display.
- [ ] UI clearly labels output as educational analysis, not financial advice.

## Week 9 — Performance optimization and research improvements

### What to optimize next
1. Tune hyperparameters using cross-validation only (`GridSearchCV` or `RandomizedSearchCV`) while keeping a final test set untouched.
2. Compare macro F1 and class-wise recall, not just accuracy.
3. Examine whether class boundaries are stable if thresholds change from 30/70 to 25/75 or 20/80.
4. Check feature-importance stability across folds or use permutation importance.
5. Improve the dataset with dated historical observations and a target based on future outcomes.

### To make a genuine future-return prediction study
- collect timestamped historical fundamentals and prices;
- define the prediction date and future horizon (e.g., future 3-month total return);
- avoid features published after the prediction date;
- define a defensible target (for example, return buckets or risk-adjusted excess return);
- split chronologically into train, validation and test periods;
- compare against simple baselines and account for transaction costs and survivorship bias.

The current workbook is a cross-sectional snapshot and lacks dated future returns, so it cannot support that stronger claim.

In [ ]:
# Cell 43 — Optional sensitivity analysis for the heuristic label thresholds
# This experiment changes only the class thresholds and reports the class counts.
# It does not retrain models, and it does not identify a financially correct threshold.
sensitivity_rows = []
for lower_q, upper_q in [(0.20, 0.80), (0.25, 0.75), (0.30, 0.70), (0.35, 0.65)]:
    low_cut = model_base["Financial_Score"].quantile(lower_q)
    high_cut = model_base["Financial_Score"].quantile(upper_q)
    labels = np.where(
        model_base["Financial_Score"] <= low_cut, "Overvalued",
        np.where(model_base["Financial_Score"] >= high_cut, "Undervalued", "Fairly Valued")
    )
    counts = pd.Series(labels).value_counts()
    sensitivity_rows.append({
        "Lower_Quantile": lower_q,
        "Upper_Quantile": upper_q,
        "Overvalued_Count": int(counts.get("Overvalued", 0)),
        "Fairly_Valued_Count": int(counts.get("Fairly Valued", 0)),
        "Undervalued_Count": int(counts.get("Undervalued", 0)),
    })
sensitivity_df = pd.DataFrame(sensitivity_rows)
display(sensitivity_df)
sensitivity_df.to_csv("Week9_Label_Threshold_Sensitivity.csv", index=False)

## Week 10 — Documentation, report and presentation

### Recommended final report structure
1. Abstract and keywords
2. Introduction and problem statement
3. Literature survey and research gap
4. Objectives and scope
5. Dataset description and data limitations
6. Preprocessing and exploratory analysis
7. Feature engineering and financial scoring
8. Random Forest methodology
9. AdaBoost methodology
10. XGBoost methodology
11. Experimental setup and evaluation metrics
12. Results and discussion (use the actual generated metrics)
13. Application architecture and integration plan
14. Limitations, responsible use and future work
15. Conclusion and references

### Results paragraph template
“On the held-out 20% split, **[model]** achieved the highest **[chosen metric]** among the three models. The macro F1 score was considered alongside accuracy to assess performance across all classes. These results quantify agreement with the heuristic labels generated from the same financial indicators; they should not be interpreted as evidence of future stock-return prediction. The dataset is a static workbook snapshot and does not include dated future returns.”

Replace the bracketed fields using the actual outputs after running the notebook.

### Viva preparation — questions you should be able to answer

1. Why did the original script use FMP, and why does this notebook use Excel?
2. Why can one workbook have repeated headers, and how does the parser detect them?
3. Why is missing data not automatically replaced with zero?
4. What is the difference between P/E, price-to-book, ROE, ROA, OPM and debt-to-equity?
5. Why are features converted to percentile ranks before aggregation?
6. Why is the score called a heuristic rather than intrinsic valuation?
7. What does a stratified train/test split preserve?
8. Why is median imputation inside a pipeline?
9. How does Random Forest differ from AdaBoost?
10. How does XGBoost differ from Random Forest and AdaBoost?
11. What is the difference between weighted F1 and macro F1?
12. What does a confusion matrix reveal?
13. Why is cross-validation useful?
14. Why can feature importance be misleading when features are correlated?
15. Why does high accuracy here not prove future stock performance?
16. What extra data would be needed for future-return prediction?
17. Why should the final pipeline, label encoder and feature list be saved together?
18. How would you expose the trained model through a backend API?
19. How should Gemini explanations be constrained to avoid inventing financial facts?
20. What security risk existed in the original source file?

### Deliverables created by this notebook
- `Week2_Parsed_Screener_Data.csv`
- `Week2_Schema_Coverage_Report.csv`
- `Week3_Missingness_Report.csv`
- `Week3_IQR_Outlier_Report.csv`
- `Week3_Correlation_Matrix.csv` (if sufficient features exist)
- `Week4_Feature_Audit.csv`
- `Week4_Financial_Data_With_Scores.csv`
- `Week4_Ranked_Company_Scores.csv`
- `Week5_Model_Comparison_Test_Metrics.csv`
- `Week5_Model_Comparison_Cross_Validation.csv` (if possible)
- `Week5_Feature_Importance.csv`
- `Week5_Test_Set_Predictions.csv`
- `Week5_All_Companies_Model_Predictions.csv`
- `finlearn_artifacts/` with all three model pipelines, class encoder, feature names and metrics
- Week 8 validation report and Week 9 threshold sensitivity report

## Appendix — Limitations to state explicitly

1. **Different source and market universe:** the original code calls Financial Modeling Prep for a list of mostly US tickers; the supplied Excel workbook is a Screener snapshot with company names and different financial fields. The notebook adapts the old workflow to the supplied workbook rather than pretending the datasets are identical.
2. **Incomplete feature equivalence:** Revenue Growth, Current Ratio, Net Income Growth and Net Profit Margin from the original 12-feature design are not available with reliable coverage in this workbook. The code uses eligible columns that really exist.
3. **Snapshot and provenance:** the workbook does not provide a usable observation date for each row; duplicate handling retains the first occurrence and cannot claim it is the newest record.
4. **Heuristic target:** class labels are created from directional ranks of the financial features. The models learn to approximate those labels, not independently observed analyst valuations.
5. **No future-return target:** the workbook does not include future price returns or dated historical data, so this is not a backtested trading strategy.
6. **Finance context matters:** a low multiple, high growth rate or low debt ratio is not always favorable. Industry comparisons, accounting quality, cash flow, cyclicality and risk matter.
7. **Application scope:** model artifacts and an inference helper are included; a deployed backend, React UI and Gemini integration are separate development work.
8. **API secret:** the original Python file contained a hard-coded API key. Rotate it if real and do not commit credentials to GitHub.

## Final completion checklist

- [ ] I ran all cells from top to bottom without skipping setup cells.
- [ ] I inspected the feature audit and understand why some original features are unavailable.
- [ ] I saved and reviewed the missingness/outlier/correlation reports.
- [ ] I can explain the score formula and label thresholds.
- [ ] I compared Random Forest, AdaBoost and XGBoost using the same holdout split.
- [ ] I interpreted class-wise reports and confusion matrices.
- [ ] I saved `finlearn_artifacts/` and downloaded the generated CSVs.
- [ ] I wrote the Week 3 observations and Week 5 interpretation in my own words.
- [ ] My report explicitly states the rule-label and no-future-return limitations.
- [ ] I have not included any API keys in a public notebook or repository.

**Project status:** the notebook implements and evaluates the three-model ML workflow. The complete product still requires the separately planned backend, Gemini integration, React frontend and end-to-end deployment/testing.